<a id="Estimation"></a>
# Snowflake Estimation

MyNote: These are just snowpark versions of functions available as SQL funcs, and they use python time to record time to run functions. I haven't actually run this.

<a id="topics"></a>
### Topics in this lesson

1. [Estimation Functions](#Estimation_Functions)  
    1. [Distinct Count](#Distinct_Count)  
    1. [Percentiles](#Percentiles)  
    1. [Other functions](#Other_functions)  


<a id="Initial_setup"></a>
### Initial Setup

#### Connect and create a `Session`

The following cell connects to your Snowflake account and creates an instance of `Session`. 

*You needn't modify anything in this cell. Just run it.*
> &#10071; Success requires that you have already completed the key pair authentication exercise.

In [ ]:
# Run utils notebook
%run ../../utils/ds_utils_python.ipynb

# Connect to Snowflake and create a Session object named session
session = create_session()

---
#### Setup Code

We should set ourselves up for success. The following ensures our context is set properly for database, schema, role, and warehouse.

*You needn't edit anything in the following cell. Just run it.*

In [ ]:
# Hard code the lesson name
lesson_name = "SNOWPARK_ESTIMATION_PY"

# Create the context items for this lesson
lesson = confirm_or_create_lesson_context(session, lesson_name)

In [ ]:
from snowflake.snowpark.functions import approx_count_distinct, countDistinct, median, approx_percentile

<a id="Estimation_Functions"></a>
## 1. Estimation Functions

In Snowflake there are several functions for approximate calculations.

These are:
- Time-efficient: To get results faster than exact calculations
- Use only when approximated results are appropriate

They are used primarily on large datasets if exact calculations are:
- Time-consuming
- Computationally intensive

Snowflake has functions for the estimation of:
- Cardinality – https://docs.snowflake.com/en/user-guide/querying-approximate-cardinality 

- Percentile – https://docs.snowflake.com/en/user-guide/querying-approximate-percentile-values

- Similarity – https://docs.snowflake.com/en/user-guide/querying-approximate-similarity

- Frequency –  https://docs.snowflake.com/en/user-guide/querying-approximate-frequent-values


To be able to estimate, Snowflake uses the HyperLogLog algorithm. 

This algorithm is known to provide accurate estimations in less time. 

For example, we use HyperLogLog to estimate a data set's approximate number of distinct values.

Let's take a look at some of these functions:

In [ ]:
# We sample from a table called customer_loyalty
full_table_name = "tasty_bytes.raw_customer.customer_loyalty"
full_table_name = "TRAINING_DB.TPCH_SF10.lineitem"
full_table_name = "SNOWFLAKE_SAMPLE_DATA.TPCH_SF100.lineitem"

df = session.table(full_table_name)

In [ ]:
# This table contains about 600 million rows
df.count()

<a id="Distinct_Count"></a>
### 1.1 Distinct Count

Let's compare the distinct count with the approximation

In [ ]:
# Turn off caching
session.sql("ALTER SESSION SET USE_CACHED_RESULT = FALSE").show()

# For testing we will suspend the warehouse every time
curr_wh = session.get_current_warehouse()
print(curr_wh)

In [ ]:
# Attempt to suspend the warehouse
try:
    # Suspending a warehouse that is not resumed can raise and exception
    session.sql(f"ALTER WAREHOUSE {curr_wh} SUSPEND").collect()
    print("Warehouse is now offline")
except Exception as ex:
    pass # Ignore the exception


In [ ]:
# Normal distinct count
import time
before = time.time()


df.select(countDistinct("L_COMMENT").alias("result")).show()

# Note the time after our action and calculate how long the action took to complete
duration_distinct_count = time.time() - before

# Print the performance duration
normal_duration_report = f"Distinct count:\nCounting rows from took {duration_distinct_count:.2f} seconds."
print(normal_duration_report)


In [ ]:
# Attempt to suspend the warehouse
try:
    # Suspending a warehouse that is not resumed can raise and exception
    session.sql(f"ALTER WAREHOUSE {curr_wh} SUSPEND").collect()    
    print("Warehouse is now offline")
except Exception as ex:
    pass # Ignore the exception


In [ ]:
# Approximate distinct count
import time
before = time.time()


df.select(approx_count_distinct("L_COMMENT").alias("result")).show()

# Note the time after our action and calculate how long the action took to complete
duration_approx_distinct_count = time.time() - before

# Print the performance duration
duration_report = f"Approximate Distinct count:\nCounting rows from took {duration_approx_distinct_count:.2f} seconds."
print(duration_report)


In [ ]:
print(normal_duration_report)
print(duration_report)

<a id="Percentiles"></a>
### 1.2 Percentiles

Next let's take a look at percentiles.

As we know calculations of percentiles can take a long time. 

We will compare the median with an approximation of the median, which is a percentile of 0.5

In [ ]:
# Attempt to suspend the warehouse
try:
    # Suspending a warehouse that is not resumed can raise and exception
    session.sql(f"ALTER WAREHOUSE {curr_wh} SUSPEND").collect()    
    print("Warehouse is now offline")
except Exception as ex:
    pass # Ignore the exception


In [ ]:
# Normal median calculation
import time
before = time.time()


df.select(median("L_SUPPKEY").alias("result")).show()

# Note the time after our action and calculate how long the action took to complete
duration_median = time.time() - before

# Print the performance duration
median_duration_report = f"Calculating the median took {duration_median:.2f} seconds."
print(median_duration_report)

In [ ]:
# Attempt to suspend the warehouse
try:
    # Suspending a warehouse that is not resumed can raise and exception
    session.sql(f"ALTER WAREHOUSE {curr_wh} SUSPEND").collect()    
    print("Warehouse is now offline")
except Exception as ex:
    print("Warehouse was already offline")
    pass # Ignore the exception


In [ ]:
# Normal distinct count
import time
before = time.time()


df.select(approx_percentile("L_SUPPKEY", 0.5).alias("result")).show()

# Note the time after our action and calculate how long the action took to complete
duration_approx_median = time.time() - before

# Print the performance duration
approx_duration_report = f"Calculating the approximate median took {duration_approx_median:.2f} seconds."
print(approx_duration_report)

In [ ]:
print(median_duration_report)
print(approx_duration_report)

<a id="Other_functions"></a>
### 1.3 Other functions

There are more functions that use HLL.

**Cardinality**
- APPROX_COUNT_DISTINCT
- HLL
- HLL_ACCUMULATE
- HLL_COMBINE
- HLL_ESTIMATE
- HLL_EXPORT
- HLL_IMPORT

**Percentile**
- APPROX_PERCENTILE
- APPROX_PERCENTILE_ACCUMULATE
- APPROX_PERCENTILE_COMBINE
- APPROC_PERCENTILE_ESTIMATE

**Similarity**
- APPROXIMATE_JACCARD_INDEX
- APPROXIMATE_SIMILARITY
- MINHASH
- MINHASH_COMBINE

**Frequency**
- APPROX_TOP_K
- APPROX_TOP_K_ACCUMULATE
- APPROX_TOP_K_COMBINE
- APPROX_TOP_K_ESTIMATE


Check the documentation if you want to know more about these functions.

In [ ]:
close_session_and_clean_up(get_lesson())

### &#10071; `Shut Down Kernel`
> After completing the activities in a notebook and before moving on to the next exercise, shut down the completed notebook by right-clicking on the notebook name and selecting `Shut Down Kernel`.